# 05_finbert_tone

Force repo root

In [2]:
from pathlib import Path
import os

p = Path.cwd().resolve()
while not (p / ".git").exists() and p != p.parent:
    p = p.parent
os.chdir(p)

print("CWD set to repo root:", Path.cwd())


CWD set to repo root: C:\Users\Andre\iCloud Drive_new\iCloudDrive\Desktop\Thesis\thesis-vrp-fomc


Load meetings and locate statement text files

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

meet = pd.read_parquet("data/processed/meetings_vrp_usmpd_tone.parquet")
meet["Date"] = pd.to_datetime(meet["Date"])
meet["event_date"] = pd.to_datetime(meet["event_date"])

TEXT_DIR = Path("data/processed/fomc_statements_text")
assert TEXT_DIR.exists(), f"Missing {TEXT_DIR}. Did Notebook 03 write statement .txt files?"

# quick check
sample_ymd = meet["Date"].iloc[0].strftime("%Y%m%d")
print("Example text path:", TEXT_DIR / f"{sample_ymd}.txt")


Example text path: data\processed\fomc_statements_text\20160316.txt


Load FinBERT and define a robust chunked scorer

Usage pattern the model author provides: https://huggingface.co/yiyanghkust/finbert-tone

In [4]:
# Usage pattern the model author provides

from transformers import BertTokenizer, BertForSequenceClassification

MODEL_NAME = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

import torch
device = torch.device("cpu")
model.to(device)

# Mapping helper

def infer_label_indices(model):
    id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
    pos = next((i for i, v in id2label.items() if "pos" in v), None)
    neg = next((i for i, v in id2label.items() if "neg" in v), None)
    neu = next((i for i, v in id2label.items() if "neu" in v), None)

    # Fallback for finbert-tone if labels are LABEL_0/LABEL_1/LABEL_2
    # (documented: 0=neutral, 1=positive, 2=negative)
    if pos is None or neg is None or neu is None:
        return 1, 2, 0
    return pos, neg, neu

POS_I, NEG_I, NEU_I = infer_label_indices(model)
print("id2label:", model.config.id2label)
print("Label indices (pos, neg, neu):", POS_I, NEG_I, NEU_I)

# Chunked scorer that works with BertTokenizer

import numpy as np
import torch

@torch.no_grad()
def finbert_score_long_text(text: str, max_length=512, stride=64):
    if text is None or not isinstance(text, str) or len(text.strip()) == 0:
        return {"p_pos": np.nan, "p_neg": np.nan, "p_neu": np.nan,
                "tone": np.nan, "entropy": np.nan, "conf": np.nan, "n_chunks": 0}

    # Tokenize once into token IDs WITHOUT special tokens
    ids = tokenizer.encode(text, add_special_tokens=False)

    # For BERT, we need [CLS] and [SEP]
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    # Reserve 2 spots for [CLS] and [SEP]
    chunk_len = max_length - 2
    step = max(1, chunk_len - stride)

    chunks = []
    weights = []

    for start in range(0, len(ids), step):
        piece = ids[start:start + chunk_len]
        if len(piece) == 0:
            break

        # Manually add special tokens: [CLS] ... [SEP]
        full = [cls_id] + piece + [sep_id]
        chunks.append(full)
        weights.append(len(piece))  # weight by number of real tokens

        if start + chunk_len >= len(ids):
            break

    # Pad to same length for batching
    maxL = max(len(c) for c in chunks)
    maxL = min(maxL, max_length)

    input_ids = []
    attention_mask = []
    for c in chunks:
        c = c[:max_length]
        m = [1] * len(c)
        if len(c) < maxL:
            padn = maxL - len(c)
            c = c + [pad_id] * padn
            m = m + [0] * padn
        input_ids.append(c)
        attention_mask.append(m)

    input_ids = torch.tensor(input_ids, dtype=torch.long, device=device)
    attention_mask = torch.tensor(attention_mask, dtype=torch.long, device=device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=1)  # (chunks, 3)

    w = torch.tensor(weights, dtype=torch.float32, device=device)
    w = w / w.sum()
    p = (probs * w.unsqueeze(1)).sum(dim=0).cpu().numpy()

    p_pos, p_neg, p_neu = float(p[POS_I]), float(p[NEG_I]), float(p[NEU_I])
    tone = p_pos - p_neg

    eps = 1e-12
    entropy = float(-(p * np.log(p + eps)).sum())  # ambiguity proxy (higher = more ambiguous)
    conf = float(p.max())                          # confidence proxy (lower = more ambiguous)

    return {"p_pos": p_pos, "p_neg": p_neg, "p_neu": p_neu,
            "tone": tone, "entropy": entropy, "conf": conf, "n_chunks": len(chunks)}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


id2label: {0: 'Neutral', 1: 'Positive', 2: 'Negative'}
Label indices (pos, neg, neu): 1 2 0


Quick sanity test

In [5]:
print(type(tokenizer))
print(type(model))
finbert_score_long_text("Growth is strong, but risks remain elevated.")

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>
<class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'>


{'p_pos': 1.0,
 'p_neg': 4.3608832100971995e-09,
 'p_neu': 3.917355329008387e-09,
 'tone': 0.9999999956391168,
 'entropy': 1.597791481344757e-07,
 'conf': 1.0,
 'n_chunks': 1}

See the chunks in action

In [6]:
print(finbert_score_long_text(open("data/processed/fomc_statements_text/20160316.txt").read())["n_chunks"])

2


Load meeting dates and confirm statement text directory

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

meet = pd.read_parquet("data/processed/meetings_vrp_usmpd_tone.parquet")
meet["Date"] = pd.to_datetime(meet["Date"])
meet["event_date"] = pd.to_datetime(meet["event_date"])

TEXT_DIR = Path("data/processed/fomc_statements_text")
assert TEXT_DIR.exists(), f"Missing {TEXT_DIR}"

print("Meetings:", len(meet))
print("Date range:", meet["Date"].min(), "to", meet["Date"].max())

Meetings: 78
Date range: 2016-03-16 00:00:00 to 2025-12-10 00:00:00


Score all statements

In [8]:
from tqdm.auto import tqdm

rows = []
missing_txt = 0

for d in tqdm(sorted(meet["Date"].unique()), desc="FinBERT scoring (78 statements)"):
    ymd = pd.to_datetime(d).strftime("%Y%m%d")
    path = TEXT_DIR / f"{ymd}.txt"
    if not path.exists():
        missing_txt += 1
        rows.append({
            "Date": pd.to_datetime(d),
            "ymd": ymd,
            "finbert_pos": np.nan,
            "finbert_neg": np.nan,
            "finbert_neu": np.nan,
            "finbert_tone": np.nan,
            "finbert_entropy": np.nan,
            "finbert_conf": np.nan,
            "finbert_uncert": np.nan,
            "finbert_n_chunks": 0,
        })
        continue

    text = path.read_text(encoding="utf-8", errors="ignore")
    out = finbert_score_long_text(text)

    rows.append({
        "Date": pd.to_datetime(d),
        "ymd": ymd,
        "finbert_pos": out["p_pos"],
        "finbert_neg": out["p_neg"],
        "finbert_neu": out["p_neu"],
        "finbert_tone": out["tone"],
        "finbert_entropy": out["entropy"],
        "finbert_conf": out["conf"],
        "finbert_uncert": 1.0 - out["conf"],  # easier interpretation than "conf"
        "finbert_n_chunks": out["n_chunks"],
    })

fin = pd.DataFrame(rows).sort_values("Date").reset_index(drop=True)
print("Missing .txt files:", missing_txt)
fin.head()

FinBERT scoring (78 statements):   0%|          | 0/78 [00:00<?, ?it/s]

Missing .txt files: 0


,Date,ymd,finbert_pos,finbert_neg,finbert_neu,finbert_tone,finbert_entropy,finbert_conf,finbert_uncert,finbert_n_chunks
0,2016-03-16,20160316,0.501615,0.008008,0.490377,0.493606,0.734167,0.501615,0.498385,2
1,2016-04-27,20160427,0.633066,0.000430,0.366504,0.632636,0.660638,0.633066,0.366934,2
2,2016-06-15,20160615,0.642796,0.009993,0.347211,0.632804,0.697382,0.642796,0.357204,2
3,2016-07-27,20160727,0.595347,0.004456,0.400197,0.590892,0.699374,0.595347,0.404653,2
4,2016-09-21,20160921,0.602251,0.000582,0.397168,0.601669,0.676467,0.602251,0.397749,2


Sanity checks on distribution

In [9]:
print(fin[["finbert_tone","finbert_entropy","finbert_conf","finbert_uncert","finbert_n_chunks"]].describe())

print("\nTop 5 most positive tone:")
print(fin.sort_values("finbert_tone", ascending=False)[["Date","finbert_tone","finbert_conf","finbert_n_chunks"]].head())

print("\nTop 5 most negative tone:")
print(fin.sort_values("finbert_tone", ascending=True)[["Date","finbert_tone","finbert_conf","finbert_n_chunks"]].head())

       finbert_tone  finbert_entropy  finbert_conf  finbert_uncert  \
count     78.000000        78.000000     78.000000       78.000000   
mean       0.391429         0.356128      0.829361        0.170639   
std        0.389104         0.281514      0.170444        0.170444   
min       -0.808478         0.000140      0.501615        0.000011   
25%        0.018122         0.042366      0.657734        0.006899   
50%        0.466039         0.385648      0.882689        0.117311   
75%        0.683390         0.653434      0.993101        0.342266   
max        0.999988         0.756932      0.999989        0.498385   

       finbert_n_chunks  
count         78.000000  
mean           1.487179  
std            0.503071  
min            1.000000  
25%            1.000000  
50%            1.000000  
75%            2.000000  
max            2.000000  

Top 5 most positive tone:
         Date  finbert_tone  finbert_conf  finbert_n_chunks
18 2018-06-13      0.999988      0.999989       

Save features files

In [10]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
fin.to_parquet("data/processed/fomc_statements_finbert.parquet", index=False)
print("Saved: data/processed/fomc_statements_finbert.parquet")

Saved: data/processed/fomc_statements_finbert.parquet


Rebuild controls and merge FinBERT

In [11]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

meet_base = pd.read_parquet("data/processed/meetings_vrp_usmpd_tone.parquet")
meet_base["Date"] = pd.to_datetime(meet_base["Date"])
meet_base["event_date"] = pd.to_datetime(meet_base["event_date"])

daily = pd.read_parquet("data/processed/vrp_vix_daily.parquet")
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.set_index("date").sort_index()

h = 22
daily["rv_back_22"] = daily["ret2"].rolling(h).sum().shift(1) * (252 / h)
daily["iv_lag1"] = daily["iv"].shift(1)

meetF = meet_base.merge(
    daily[["iv_lag1","rv_back_22"]].reset_index().rename(columns={"date":"event_date"}),
    on="event_date",
    how="left"
).merge(
    fin[["Date","finbert_tone","finbert_entropy","finbert_uncert","finbert_conf","finbert_n_chunks"]],
    on="Date",
    how="left"
)

print(meetF[["vrp","surprise_pc1","tone_net","unc_share","iv_lag1","rv_back_22",
             "finbert_tone","finbert_entropy","finbert_uncert"]].isna().sum())

print("\nCorr(LM tone, FinBERT tone):")
print(meetF[["tone_net","finbert_tone"]].corr())

vrp                0
surprise_pc1       0
tone_net           0
unc_share          0
iv_lag1            0
rv_back_22         0
finbert_tone       0
finbert_entropy    0
finbert_uncert     0
dtype: int64

Corr(LM tone, FinBERT tone):
              tone_net  finbert_tone
tone_net       1.00000       0.51165
finbert_tone   0.51165       1.00000


Regressions (HC1) and export to LaTeX

In [12]:
def ols_hc1(df, y, xcols):
    X = sm.add_constant(df[xcols].astype(float))
    yv = df[y].astype(float)
    return sm.OLS(yv, X, missing="drop").fit(cov_type="HC1")

controls = ["iv_lag1","rv_back_22"]

m1 = ols_hc1(meetF, "vrp", ["surprise_pc1", "finbert_tone"] + controls)
m2 = ols_hc1(meetF, "vrp", ["surprise_pc1", "finbert_entropy"] + controls)
m3 = ols_hc1(meetF, "vrp", ["surprise_pc1", "finbert_uncert"] + controls)

# “Incremental info” test: include both LM and FinBERT tone
m4 = ols_hc1(meetF, "vrp", ["surprise_pc1", "tone_net", "finbert_tone"] + controls)

tab = summary_col(
    [m1, m2, m3, m4],
    stars=True,
    float_format="%.4f",
    model_names=[
        "VRP~Surp+FinBERTTone+Ctrls",
        "VRP~Surp+Entropy+Ctrls",
        "VRP~Surp+Uncert+Ctrls",
        "VRP~Surp+LM+FinBERT+Ctrls"
    ],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)

print(tab)

from pathlib import Path
Path("reports/tables").mkdir(parents=True, exist_ok=True)
Path("reports/tables/reg_vrp_finbert_controls.tex").write_text(tab.as_latex(), encoding="utf-8")
print("Saved: reports/tables/reg_vrp_finbert_controls.tex")


                VRP~Surp+FinBERTTone+Ctrls VRP~Surp+Entropy+Ctrls VRP~Surp+Uncert+Ctrls VRP~Surp+LM+FinBERT+Ctrls
-----------------------------------------------------------------------------------------------------------------
const           0.0113**                   0.0029                 0.0026                0.0101**                 
                (0.0052)                   (0.0081)               (0.0073)              (0.0048)                 
surprise_pc1    0.0007                     0.0009                 0.0008                0.0006                   
                (0.0017)                   (0.0017)               (0.0016)              (0.0015)                 
finbert_tone    -0.0231***                                                              -0.0290*                 
                (0.0082)                                                                (0.0155)                 
iv_lag1         0.3289*                    0.3992**               0.3986**             

Add caption automatically

In [13]:
from pathlib import Path

def add_caption_label(tex_path, caption, label):
    tex = Path(tex_path).read_text(encoding="utf-8")
    tex = tex.replace(r"\caption{}", rf"\caption{{{caption}}}")
    tex = tex.replace(r"\label{}",   rf"\label{{{label}}}")
    Path(tex_path).write_text(tex, encoding="utf-8")

add_caption_label(
    "reports/tables/reg_vrp_finbert_controls.tex",
    "Optional extension: Meeting-level regressions using FinBERT-based statement tone and ambiguity proxies.",
    "tab:meet_finbert"
)

### Two robustness checks

Standardized FinBERT tone

In [14]:
meetF["finbert_tone_z"] = (meetF["finbert_tone"] - meetF["finbert_tone"].mean()) / meetF["finbert_tone"].std()
m_z = ols_hc1(meetF, "vrp", ["surprise_pc1", "finbert_tone_z", "iv_lag1", "rv_back_22"])
print(m_z.summary())

                            OLS Regression Results                            
Dep. Variable:                    vrp   R-squared:                       0.151
Model:                            OLS   Adj. R-squared:                  0.104
Method:                 Least Squares   F-statistic:                     4.798
Date:                Sat, 21 Mar 2026   Prob (F-statistic):            0.00171
Time:                        19:01:17   Log-Likelihood:                 167.26
No. Observations:                  78   AIC:                            -324.5
Df Residuals:                      73   BIC:                            -312.7
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0023      0.004      0.

Influence check (I could possibly delete the cell below)

In [15]:
import statsmodels.api as sm
X = sm.add_constant(meetF[["surprise_pc1","finbert_tone","iv_lag1","rv_back_22"]].astype(float))
y = meetF["vrp"].astype(float)
m = sm.OLS(y, X, missing="drop").fit()
infl = m.get_influence()
cook = infl.cooks_distance[0]
print("Top 5 Cook's distance:")
print(pd.Series(cook, index=meetF["Date"]).sort_values(ascending=False).head())

Top 5 Cook's distance:
Date
2025-05-07    0.522455
2025-03-19    0.273838
2021-01-27    0.089284
2022-05-04    0.066537
2024-12-18    0.038008
dtype: float64


### Influence robustness now

Compute Cook's distance on the same RHS variables

In [16]:
import statsmodels.api as sm
import pandas as pd
import numpy as np

Xcols = ["surprise_pc1", "finbert_tone_z", "iv_lag1", "rv_back_22"]

df = meetF.copy()
df = df.dropna(subset=["vrp"] + Xcols).copy()

X = sm.add_constant(df[Xcols].astype(float))
y = df["vrp"].astype(float)

ols = sm.OLS(y, X).fit()
cooks = ols.get_influence().cooks_distance[0]
df["cooks"] = cooks

print("4/n threshold:", 4/len(df))
print(df.sort_values("cooks", ascending=False)[["Date","cooks"]].head(10))

4/n threshold: 0.05128205128205128
         Date     cooks
72 2025-05-07  0.522455
71 2025-03-19  0.273838
38 2021-01-27  0.089284
48 2022-05-04  0.066537
69 2024-12-18  0.038008
51 2022-09-21  0.034622
31 2020-01-29  0.029364
34 2020-07-29  0.027256
32 2020-04-29  0.026303
36 2020-11-05  0.019533


Function to refit with HC1

In [17]:
def fit_hc1(d):
    X = sm.add_constant(d[Xcols].astype(float))
    y = d["vrp"].astype(float)
    return sm.OLS(y, X, missing="drop").fit(cov_type="HC1")

Drop top 1, top 2, top 5 influential points and compare

In [18]:
base = fit_hc1(df)

rows = []
rows.append(("Baseline", len(df), base.params["finbert_tone_z"], base.bse["finbert_tone_z"], base.pvalues["finbert_tone_z"], base.rsquared))

for k in [1, 2, 5]:
    d2 = df.sort_values("cooks", ascending=False).iloc[k:].copy()
    res = fit_hc1(d2)
    rows.append((f"Drop top {k} Cook's D", len(d2), res.params["finbert_tone_z"], res.bse["finbert_tone_z"], res.pvalues["finbert_tone_z"], res.rsquared))

rob = pd.DataFrame(rows, columns=["Spec","N","beta_tone_z","se","p","R2"])
rob

,Spec,N,beta_tone_z,se,p,R2
0,Baseline,78,-0.008988,0.003185,0.004777,0.150779
1,Drop top 1 Cook's D,77,-0.009248,0.003266,0.004634,0.173043
2,Drop top 2 Cook's D,76,-0.007393,0.002637,0.005061,0.279007
3,Drop top 5 Cook's D,73,-0.005565,0.002211,0.011854,0.373336


Export LaTeX robustness table

In [19]:
from pathlib import Path

Path("reports/tables").mkdir(parents=True, exist_ok=True)
rob.to_latex("reports/tables/finbert_influence_robustness.tex", index=False, float_format="%.4f")
print("Saved: reports/tables/finbert_influence_robustness.tex")

Saved: reports/tables/finbert_influence_robustness.tex


Leave-one-out range of the coefficient

In [20]:
betas = []
pvals = []
dates = []

for i in range(len(df)):
    d_loo = df.drop(df.index[i])
    res = fit_hc1(d_loo)
    betas.append(res.params["finbert_tone_z"])
    pvals.append(res.pvalues["finbert_tone_z"])
    dates.append(df.loc[df.index[i], "Date"])

betas = pd.Series(betas, index=pd.to_datetime(dates))
pvals = pd.Series(pvals, index=pd.to_datetime(dates))

print("LOO beta_tone_z summary:")
print(betas.describe())

print("\nWorst 5 (most negative):")
print(betas.sort_values().head())

print("\nBest 5 (closest to 0):")
print(betas.sort_values().tail())

LOO beta_tone_z summary:
count    78.000000
mean     -0.008988
std       0.000372
min      -0.009788
25%      -0.009147
50%      -0.009036
75%      -0.008963
max      -0.007175
dtype: float64

Worst 5 (most negative):
2020-07-29   -0.009788
2019-01-30   -0.009763
2019-06-19   -0.009486
2024-07-31   -0.009420
2019-12-11   -0.009410
dtype: float64

Best 5 (closest to 0):
2018-09-26   -0.008493
2024-12-18   -0.008330
2020-01-29   -0.007883
2021-01-27   -0.007754
2025-03-19   -0.007175
dtype: float64


Optional appendix: Standardized FinBERT tone regression (HC1)

In [21]:
from statsmodels.iolib.summary2 import summary_col
from pathlib import Path

m_z = ols_hc1(meetF, "vrp", ["surprise_pc1","finbert_tone_z","iv_lag1","rv_back_22"])

tab_z = summary_col(
    [m_z],
    stars=True,
    float_format="%.4f",
    model_names=["VRP~Surp+FinBERTTone(z)+Ctrls"],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)

Path("reports/tables").mkdir(parents=True, exist_ok=True)
Path("reports/tables/reg_vrp_finbert_standardized.tex").write_text(tab_z.as_latex(), encoding="utf-8")
print("Saved: reports/tables/reg_vrp_finbert_standardized.tex")

Saved: reports/tables/reg_vrp_finbert_standardized.tex
